# Representation screen, probe protocol: labscalars

Flags beyond the base: `--lab-scalars`. Seeds 0, 1, 2, K=32 / 400k jets / 20 epochs, T4 with
torch 2.11.0+cu128 pinned, paired seed by seed with the 2026-10-05 control (`results_message/probe_screen/base`). Kernel budget 1.75 h.

In [ ]:
import subprocess, time
T0 = time.time()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())

In [ ]:
import subprocess, sys, time
WANT = "2.11.0+cu128"
name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("device:", name)
assert "T4" in name, "paired with T4 runs only; got %r -- stopping before any GPU time is spent" % name

def torch_version():
    return subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                          capture_output=True, text=True).stdout.strip()

v = torch_version()
if v != WANT:
    print("torch", v, "- installing", WANT)
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.11.0",
                        "--index-url", "https://download.pytorch.org/whl/cu128"],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, "in %.0f s" % (time.perf_counter() - t0))
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit("could not install torch " + WANT)
    v = torch_version()
assert v == WANT, "torch %s, expected %s" % (v, WANT)
print("torch", v, "on", name)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchdiffeq"], check=True)

In [ ]:
import base64, io, tarfile, pathlib
CODE_B64 = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode="r:gz").extractall("/kaggle/working/repo")
print(sorted(p.name for p in pathlib.Path("/kaggle/working/repo").iterdir()))

In [ ]:
import subprocess, sys

def run(args, cwd="/kaggle/working/repo"):
    """Run a repo module, stream nothing, return (ok, tail-of-output)."""
    r = subprocess.run([sys.executable, "-m"] + args, cwd=cwd,
                       capture_output=True, text=True)
    out = r.stdout[-4000:]
    if r.returncode != 0:
        out += "\n--- stderr ---\n" + r.stderr[-3000:]
    print(out)
    return r.returncode == 0

In [ ]:
import glob, pathlib
cands = glob.glob("/kaggle/input/**/top_tagging_train.npz", recursive=True)
assert cands, "attach the K=128 data-prep kernel output"
DATA = str(pathlib.Path(cands[0]).parent)
TAG, SEEDS = 'labscalars', [0, 1, 2]
OUT = "/kaggle/working/results_probe_screen/" + TAG
BASE = ['--cache-dir', 'DATA', '--representation', 'constituents', '--canonical-splits', '--normalize', 'global', '--max-train-samples', '400000', '--epochs', '20', '--device', 'cuda', '--dtype', 'float32', '--batch-size', '256', '--n-constituents', '32', '--eval-chunk-size', '2048', '--models', 'so3c_message_set', '--rounds', '3', '--beams', '--channels', '8', '--compile', '--lab-scalars']
BASE[BASE.index("DATA")] = DATA
BUDGET = 1.75 * 3600           # this kernel's share of the week's GPU quota
EVAL_MARGIN = 240            # final test evaluation and writing results
MIN_SEED = 1500              # a seed needs ~0.45 h; less than this cannot finish
pathlib.Path(OUT).mkdir(parents=True, exist_ok=True)
pathlib.Path(OUT, "bundle_sha256.txt").write_text('638abc237aa6' + chr(10))
print("data:", DATA, "| bundle", '638abc237aa6')

In [ ]:
# The control this arm pairs with (results_message/probe_screen/base): per seed, (test AUC,
# rejection at 30%); seed 0's (train, val) loss in epochs 1 and 2; and its
# command, which is BASE without this arm's flags.
CONTROL_TEST = {0: (0.9811491732570162, 804.4382470119522), 1: (0.9811092523841787, 824.138775510204), 2: (0.981106801276, 824.138775510204)}
CONTROL_HISTORY = [(0.22157232296466828, 0.20258820056915283), (0.19644562244415284, 0.1904924362897873)]
CONTROL_BASE = ['--cache-dir', 'DATA', '--representation', 'constituents', '--canonical-splits', '--normalize', 'global', '--max-train-samples', '400000', '--epochs', '20', '--device', 'cuda', '--dtype', 'float32', '--batch-size', '256', '--n-constituents', '32', '--eval-chunk-size', '2048', '--models', 'so3c_message_set', '--rounds', '3', '--beams', '--channels', '8', '--compile']
CONTROL_BASE[CONTROL_BASE.index("DATA")] = DATA

In [ ]:
# On-card guard: the forward pass must be the published function here too.
import subprocess, sys
g = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                    "tests/test_so3c_models.py", "-k", "golden_cuda"],
                   cwd="/kaggle/working/repo", capture_output=True, text=True)
print(g.stdout[-1200:], g.stderr[-800:])
assert "1 passed" in g.stdout, "golden guard did not pass on the card"

In [ ]:
import glob, time
for seed in SEEDS:
    left = BUDGET - (time.time() - T0) - EVAL_MARGIN
    print("=== seed %d: %.2f h of kernel budget left ===" % (seed, left / 3600), flush=True)
    if left < MIN_SEED:
        print("not enough budget left; seed %d skipped" % seed)
        continue
    run(["benchmarks.run_top_tagging"] + BASE +
        ["--seed", str(seed), "--results-dir", OUT + "/seed%d" % seed,
         "--ckpt-dir", "/kaggle/working/checkpoints/%s_seed%d" % (TAG, seed),
         "--resume", "--max-seconds", str(int(left))])
    assert glob.glob(OUT + "/seed%d/*.json" % seed), "seed %d wrote no result" % seed
print("kernel time %.2f h" % ((time.time() - T0) / 3600))

In [ ]:
import json, glob, statistics
print("%-6s%9s%10s%9s%8s%8s%7s%14s" % ("seed", "AUC", "dAUC", "rej@0.3", "drej",
                                        "epochs", "hours", "batch order"))
deltas, drejs, orders = {}, {}, {}
for seed in SEEDS:
    for f in sorted(glob.glob(OUT + "/seed%d/*.json" % seed)):
        r = json.load(open(f)); t = r["test_metrics"]; env = r.get("env", {})
        c_auc, c_rej = CONTROL_TEST[seed]
        orders[seed] = r["batch_order"]["epoch1"]
        print("%-6d%9.5f%+10.5f%9.1f%+8.1f%8d%7.2f%14s"
              % (seed, t["test_auc"], t["test_auc"] - c_auc, t["bg_rej_30"],
                 t["bg_rej_30"] - c_rej, r["epochs_run"], r["walltime_sec"] / 3600,
                 orders[seed][:12]))
        assert env.get("torch") == "2.11.0+cu128" and "T4" in env.get("gpu", ""), env
        if r["epochs_run"] < 20:
            print("  INCOMPLETE: stopped at epoch %d on the kernel budget" % r["epochs_run"])
        else:
            deltas[seed] = t["test_auc"] - c_auc
            drejs[seed] = t["bg_rej_30"] - c_rej
if "CHECK_ORDER" in globals() and 0 in orders:
    same = orders[0] == CHECK_ORDER["epoch1"]
    print("seed 0 batch order %s the pairing check's" % ("equals" if same else "DIFFERS FROM"))
d = [deltas[s] for s in SEEDS if s in deltas]
if len(d) < len(SEEDS):
    print("%d of %d seeds complete: no verdict" % (len(d), len(SEEDS)))
else:
    m = statistics.mean(d)
    if m >= 0.00015 and min(d) > 0:
        verdict = "PROMOTE to the K=32 long schedule"
    elif abs(m) < 0.0001 or min(d) < 0 < max(d):
        verdict = "CLOSE at the probe level"
    elif max(d) < 0:
        verdict = "UNRESOLVED by the rule, but every seed is negative: not a candidate"
    else:
        verdict = "UNRESOLVED"
    print("paired dAUC %s: mean %+.5f sd %.5f | drej mean %+.1f"
          % (" ".join("%+.5f" % x for x in d), m, statistics.stdev(d),
             statistics.mean(drejs.values())))
    print("verdict: %s" % verdict)
    print("(valid only if the pairing check in probe-screen-beame002 passed;"
          " rejection is reported, never decides)")